In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
class LoraLayer(nn.Module):
    def __init__(self, target_linear_layer, feature_in, feature_out, r, alpha,dropout=0.1):
        super(LoraLayer,self).__init__()
        # 第一步，初始化lora的一些参数，包含a矩阵，b矩阵，r秩.比例系数等等。
        self.lora_a = nn.Parameter(torch.zeros(r, feature_in))
        self.lora_b = nn.Parameter(torch.zeros(feature_out,r))
        self.alpha = alpha
        self.r = r
        self.dropout_rate = dropout
        self.scale = alpha/r
        if self.dropout_rate>0:
            self.dropout = nn.Dropout(self.dropout_rate)
        else:
            self.dropout = nn.Identity()
        # 第二步对alpha进行初始化
        nn.init.kaiming_uniform_(self.lora_a)
        # 第三步，初始化原本的目标线性层
        self.net = target_linear_layer
        self.net.weight.requires_grad=False
    def forward(self, x):
        output1 = self.net(x)
        output2 = F.linear(x, self.lora_b @ self.lora_a * self.scale)  # 得到结果后，乘上比例系数(alpha/r)
        return self.dropout(output1+output2)
·

batch_size = 32
seq_len = 128
in_features = 768
out_features = 512
rank = 8
lora_alpha = 16
dropout = 0.1

# Create a test input
x = torch.randn(batch_size, seq_len, in_features)
weight = nn.Parameter(torch.zeros(in_features,out_features))
nn.init.kaiming_uniform_(weight)
# Test regular mode (no merge)
lora_layer = LoRA(
    weight = weight,
    feature_in=in_features,
    feature_out=out_features,
    rank=rank,
    alpha=lora_alpha,
    dropout=dropout,
)
output = lora_layer(x)
print(f"Output shape (no merge): {output.shape}") 

Output shape (no merge): torch.Size([32, 128, 512])


In [ ]:
import torch
import torch.nn
import torch.nn.functional as F
import math
class LoRA(nn.Module):
    def __init__(self,weight,feature_in,feature_out,rank,alpha,dropout=0.1,merged=False):
        super(LoRA,self).__init__()
        self.feature_in = feature_in
        self.feature_out = feature_out
        self.rank = rank
        self.alpha = alpha
        self.merged = merged
        self.scale=  alpha//rank
        if dropout>0:
            self.dropout = nn.Dropout(dropout)
        else:
            self.dropout = nn.Identity()
        
        self.lora_b = nn.Parameter(torch.zeros(self.rank,self.feature_out))
        self.lora_a = nn.Parameter(torch.zeros(self.feature_in,self.rank))
        self.weight = weight
        self.weight.requires_grad = False
    
        nn.init.kaiming_uniform_(self.lora_a)
    
    def forward(self,x):
        if self.merged:
            output = x@(self.weight+self.lora_a@self.lora_b*self.scale)
        else:
            output = x@self.weight
        return self.dropout(output)
    def merge_weight(self,):
        self.weight.data += self.lora_a@self.lora_b*self.scale

batch_size = 32
seq_len = 128
in_features = 768
out_features = 512
rank = 8
lora_alpha = 16
dropout = 0.1

# Create a test input
x = torch.randn(batch_size, seq_len, in_features)
weight = nn.Parameter(torch.zeros(in_features,out_features))
nn.init.kaiming_uniform_(weight,a=math.sqrt(5))
# Test regular mode (no merge)
lora_layer = LoRA(
    weight = weight,
    feature_in=in_features,
    feature_out=out_features,
    rank=rank,
    alpha=lora_alpha,
    dropout=dropout,
)
output = lora_layer(x)
lora_layer.merge_weight()
print(f"Output shape (no merge): {output.shape}") 

Output shape (no merge): torch.Size([32, 128, 512])
